Objective 1:

In [1]:
from google.colab import files
uploaded = files.upload()

Saving kaggle.zip to kaggle.zip


In [2]:
import zipfile
import os

with zipfile.ZipFile("kaggle.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

os.listdir("data")

['mimic-iv-clinical-database-demo-2.2']

In [3]:
import pandas as pd

patients = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/patients.csv")
admissions = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/admissions.csv")
diagnoses = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/diagnoses_icd.csv")

In [4]:
df = admissions.merge(patients, on="subject_id")
df = df.merge(diagnoses, on=["subject_id", "hadm_id"])

df.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,edouttime,hospital_expire_flag,gender,anchor_age,anchor_year,anchor_year_group,dod,seq_num,icd_code,icd_version
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,2196-02-24 17:07:00,0,M,47,2196,2014 - 2016,NaN,15,42731,9
1,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,2196-02-24 17:07:00,0,M,47,2196,2014 - 2016,NaN,5,51881,9
2,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,2196-02-24 17:07:00,0,M,47,2196,2014 - 2016,NaN,12,75169,9
3,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,2196-02-24 17:07:00,0,M,47,2196,2014 - 2016,NaN,13,4254,9
4,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,2196-02-24 17:07:00,0,M,47,2196,2014 - 2016,NaN,21,2749,9


In [5]:
df['admittime'] = pd.to_datetime(df['admittime'])
df['dob'] = pd.to_datetime(df['anchor_year'])  # approximate age

df['age'] = df['admittime'].dt.year - df['anchor_year']

In [6]:
df = df[['age', 'gender', 'admission_type', 'insurance', 'icd_code']]
df.dropna(inplace=True)

In [7]:
top_codes = df['icd_code'].value_counts().nlargest(5).index
df = df[df['icd_code'].isin(top_codes)]

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['gender'] = le.fit_transform(df['gender'])
df['admission_type'] = le.fit_transform(df['admission_type'])
df['insurance'] = le.fit_transform(df['insurance'])
df['icd_code'] = le.fit_transform(df['icd_code'])

In [9]:
import torch
from sklearn.model_selection import train_test_split

X = df[['age', 'gender', 'admission_type', 'insurance']].values
y = df['icd_code'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [10]:
import torch.nn as nn

class EHRModel(nn.Module):
    def __init__(self):
        super(EHRModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 5)  # 5 disease classes
        )

    def forward(self, x):
        return self.net(x)

model = EHRModel()

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 1.6480965614318848
Epoch 2, Loss: 1.6371642351150513
Epoch 3, Loss: 1.6272289752960205
Epoch 4, Loss: 1.6183876991271973
Epoch 5, Loss: 1.610248327255249
Epoch 6, Loss: 1.6027487516403198
Epoch 7, Loss: 1.5957144498825073
Epoch 8, Loss: 1.5889569520950317
Epoch 9, Loss: 1.5825092792510986
Epoch 10, Loss: 1.5761929750442505
Epoch 11, Loss: 1.569932222366333
Epoch 12, Loss: 1.5637900829315186
Epoch 13, Loss: 1.557509422302246
Epoch 14, Loss: 1.5512588024139404
Epoch 15, Loss: 1.545228362083435
Epoch 16, Loss: 1.5392677783966064
Epoch 17, Loss: 1.5332212448120117
Epoch 18, Loss: 1.5273548364639282
Epoch 19, Loss: 1.5215568542480469
Epoch 20, Loss: 1.5159423351287842


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [13]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [14]:
class EHRModel(nn.Module):
    def __init__(self):
        super(EHRModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.net(x)

In [15]:
for epoch in range(30):
    total_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 1.48955956527165
Epoch 2, Loss: 1.4778568404061454
Epoch 3, Loss: 1.4436760289328439
Epoch 4, Loss: 1.4312626464026315
Epoch 5, Loss: 1.4064727681023734
Epoch 6, Loss: 1.396000589643206
Epoch 7, Loss: 1.3749222925731115
Epoch 8, Loss: 1.3718125309262956
Epoch 9, Loss: 1.3580643108912878
Epoch 10, Loss: 1.3545899902071272
Epoch 11, Loss: 1.3499713284628732
Epoch 12, Loss: 1.3485120364597865
Epoch 13, Loss: 1.342287881033761
Epoch 14, Loss: 1.334017242704119
Epoch 15, Loss: 1.337415371622358
Epoch 16, Loss: 1.3313085011073522
Epoch 17, Loss: 1.3278164522988456
Epoch 18, Loss: 1.31532883644104
Epoch 19, Loss: 1.3101614202771867
Epoch 20, Loss: 1.2980729000908988
Epoch 21, Loss: 1.3015799522399902
Epoch 22, Loss: 1.3071095943450928
Epoch 23, Loss: 1.284789800643921
Epoch 24, Loss: 1.3100721665791102
Epoch 25, Loss: 1.296538966042655
Epoch 26, Loss: 1.2778447355542863
Epoch 27, Loss: 1.2775622776576452
Epoch 28, Loss: 1.2712534836360387
Epoch 29, Loss: 1.2636570930480957
Epoc

In [16]:
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)

print("Accuracy:", accuracy)

Accuracy: 0.3584905660377358


In [17]:
print(df.columns)

Index(['age', 'gender', 'admission_type', 'insurance', 'icd_code'], dtype='object')


In [18]:
class EHRModel(nn.Module):
    def __init__(self):
        super(EHRModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

In [19]:
top_codes = df['icd_code'].value_counts().nlargest(3).index
df = df[df['icd_code'].isin(top_codes)]

In [20]:
patients = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/patients.csv")
admissions = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/admissions.csv")
diagnoses = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/diagnoses_icd.csv")

df = admissions.merge(patients, on="subject_id")
df = df.merge(diagnoses, on=["subject_id", "hadm_id"])

In [21]:
print(df.columns)

Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'gender',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'seq_num',
       'icd_code', 'icd_version'],
      dtype='object')


In [22]:
df = df[['anchor_age', 'gender', 'admission_type',
         'admission_location', 'discharge_location',
         'insurance', 'hospital_expire_flag', 'icd_code']]

In [23]:
df.rename(columns={'anchor_age': 'age'}, inplace=True)

In [24]:
df.dropna(inplace=True)

In [25]:
top_codes = df['icd_code'].value_counts().nlargest(3).index
df = df[df['icd_code'].isin(top_codes)]

In [26]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

cols = ['gender', 'admission_type', 'admission_location',
        'discharge_location', 'insurance', 'icd_code']

for col in cols:
    df[col] = le.fit_transform(df[col])

In [27]:
X = df[['age', 'gender', 'admission_type',
        'admission_location', 'discharge_location',
        'insurance', 'hospital_expire_flag']].values

y = df['icd_code'].values

In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [30]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [31]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [32]:
import torch.nn as nn

class EHRModel(nn.Module):
    def __init__(self):
        super(EHRModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, 3)  # 3 classes
        )

    def forward(self, x):
        return self.net(x)

model = EHRModel()

In [33]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    total_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 1.1522968411445618
Epoch 2, Loss: 1.014670729637146
Epoch 3, Loss: 0.951108992099762
Epoch 4, Loss: 0.8787513971328735
Epoch 5, Loss: 0.8593035936355591
Epoch 6, Loss: 0.8352636098861694
Epoch 7, Loss: 0.8042396456003189
Epoch 8, Loss: 0.7855871915817261
Epoch 9, Loss: 0.7531238794326782
Epoch 10, Loss: 0.7449672520160675
Epoch 11, Loss: 0.6955662965774536
Epoch 12, Loss: 0.690863162279129
Epoch 13, Loss: 0.6829153597354889
Epoch 14, Loss: 0.6534209251403809
Epoch 15, Loss: 0.6726832687854767
Epoch 16, Loss: 0.6358560174703598
Epoch 17, Loss: 0.6119889914989471
Epoch 18, Loss: 0.5953214317560196
Epoch 19, Loss: 0.585071250796318
Epoch 20, Loss: 0.6197623312473297
Epoch 21, Loss: 0.5615810006856918
Epoch 22, Loss: 0.5535861551761627
Epoch 23, Loss: 0.5220024362206459
Epoch 24, Loss: 0.5240343809127808
Epoch 25, Loss: 0.5462164431810379
Epoch 26, Loss: 0.5347828269004822
Epoch 27, Loss: 0.49978453665971756
Epoch 28, Loss: 0.5162516683340073
Epoch 29, Loss: 0.51881802827119

In [34]:
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)

print("Accuracy:", accuracy)

Accuracy: 0.34375


In [35]:
print(df['icd_code'].value_counts())

icd_code
1    60
0    50
2    46
Name: count, dtype: int64


In [36]:
y = df['hospital_expire_flag'].values

In [37]:
nn.Linear(64, 2)  # binary classification

Linear(in_features=64, out_features=2, bias=True)

In [38]:
criterion = nn.CrossEntropyLoss()

In [39]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [40]:
for epoch in range(50):
    total_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.36333513259887695
Epoch 2, Loss: 0.3906968906521797
Epoch 3, Loss: 0.3617907017469406
Epoch 4, Loss: 0.3921678960323334
Epoch 5, Loss: 0.3692067414522171
Epoch 6, Loss: 0.3523062616586685
Epoch 7, Loss: 0.39055705070495605
Epoch 8, Loss: 0.36875028535723686
Epoch 9, Loss: 0.37528352439403534
Epoch 10, Loss: 0.45475903898477554
Epoch 11, Loss: 0.4145067185163498
Epoch 12, Loss: 0.3561433106660843
Epoch 13, Loss: 0.343487873673439
Epoch 14, Loss: 0.3783938139677048
Epoch 15, Loss: 0.34323035553097725
Epoch 16, Loss: 0.34184252470731735
Epoch 17, Loss: 0.3503616005182266
Epoch 18, Loss: 0.3460716977715492
Epoch 19, Loss: 0.34824588894844055
Epoch 20, Loss: 0.35552791506052017
Epoch 21, Loss: 0.3556569740176201
Epoch 22, Loss: 0.34465353190898895
Epoch 23, Loss: 0.34738919883966446
Epoch 24, Loss: 0.3635910004377365
Epoch 25, Loss: 0.3717334046959877
Epoch 26, Loss: 0.36704540252685547
Epoch 27, Loss: 0.36198728531599045
Epoch 28, Loss: 0.3507283106446266
Epoch 29, Loss: 0

In [41]:
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)

print("Accuracy:", accuracy)

Accuracy: 0.3125


In [42]:
print(df['hospital_expire_flag'].value_counts())

hospital_expire_flag
0    146
1     10
Name: count, dtype: int64


In [43]:
min_count = df['hospital_expire_flag'].value_counts().min()

df_balanced = df.groupby('hospital_expire_flag').apply(
    lambda x: x.sample(min_count)
).reset_index(drop=True)

df = df_balanced

/tmp/ipykernel_1564/2992378151.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('hospital_expire_flag').apply(


In [44]:
print(df['hospital_expire_flag'].value_counts())

hospital_expire_flag
0    10
1    10
Name: count, dtype: int64


In [45]:
for epoch in range(50):
    total_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.34017156064510345
Epoch 2, Loss: 0.336329884827137
Epoch 3, Loss: 0.33300747349858284
Epoch 4, Loss: 0.3367626294493675
Epoch 5, Loss: 0.3572356253862381
Epoch 6, Loss: 0.3490532264113426
Epoch 7, Loss: 0.32047659903764725
Epoch 8, Loss: 0.31467268615961075
Epoch 9, Loss: 0.3476568013429642
Epoch 10, Loss: 0.3222886025905609
Epoch 11, Loss: 0.3215976394712925
Epoch 12, Loss: 0.3272281810641289
Epoch 13, Loss: 0.38339024037122726
Epoch 14, Loss: 0.3016521818935871
Epoch 15, Loss: 0.3327556475996971
Epoch 16, Loss: 0.3225368559360504
Epoch 17, Loss: 0.3393700197339058
Epoch 18, Loss: 0.30425499379634857
Epoch 19, Loss: 0.3234869949519634
Epoch 20, Loss: 0.3681010901927948
Epoch 21, Loss: 0.34685707464814186
Epoch 22, Loss: 0.3107156828045845
Epoch 23, Loss: 0.30919981747865677
Epoch 24, Loss: 0.3209172263741493
Epoch 25, Loss: 0.3470931574702263
Epoch 26, Loss: 0.3431163653731346
Epoch 27, Loss: 0.340662632137537
Epoch 28, Loss: 0.3211449459195137
Epoch 29, Loss: 0.34324

In [46]:
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)

print("Accuracy:", accuracy)

Accuracy: 0.28125


In [47]:
print("Predicted:", predicted[:10])
print("Actual:", y_test[:10])

Predicted: tensor([0, 0, 2, 1, 1, 1, 1, 2, 1, 1])
Actual: tensor([1, 1, 2, 2, 0, 0, 0, 1, 0, 2])


In [48]:
print(df['hospital_expire_flag'].unique())

[0 1]


In [49]:
y = df['hospital_expire_flag'].values

In [50]:
nn.Linear(64, 2)  # ONLY 2 classes

Linear(in_features=64, out_features=2, bias=True)

In [51]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import TensorDataset, DataLoader

In [52]:
patients = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/patients.csv")
admissions = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/admissions.csv")
diagnoses = pd.read_csv("/content/data/mimic-iv-clinical-database-demo-2.2/hosp/diagnoses_icd.csv")

df = admissions.merge(patients, on="subject_id")
df = df.merge(diagnoses, on=["subject_id", "hadm_id"])

In [53]:
df = df[['anchor_age', 'gender', 'admission_type',
         'admission_location', 'discharge_location',
         'insurance', 'hospital_expire_flag']]

In [54]:
df.rename(columns={'anchor_age': 'age'}, inplace=True)
df.dropna(inplace=True)

In [55]:
le = LabelEncoder()

cols = ['gender', 'admission_type', 'admission_location',
        'discharge_location', 'insurance']

for col in cols:
    df[col] = le.fit_transform(df[col])

In [56]:
df = df.groupby('hospital_expire_flag', group_keys=False).apply(
    lambda x: x.sample(min_count)
).reset_index(drop=True)

/tmp/ipykernel_1564/2730303647.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('hospital_expire_flag', group_keys=False).apply(


In [57]:
X = df[['age', 'gender', 'admission_type',
        'admission_location', 'discharge_location',
        'insurance']].values

y = df['hospital_expire_flag'].values  # ✅ FINAL TARGET

In [58]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [60]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [61]:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [62]:
class EHRModel(nn.Module):
    def __init__(self):
        super(EHRModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(6, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 2)  # ✅ binary output
        )

    def forward(self, x):
        return self.net(x)

model = EHRModel()

In [63]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [64]:
for epoch in range(50):
    total_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.7219151258468628
Epoch 2, Loss: 0.8321334719657898
Epoch 3, Loss: 0.7468668818473816
Epoch 4, Loss: 0.7147054672241211
Epoch 5, Loss: 0.6457453966140747
Epoch 6, Loss: 0.6102166175842285
Epoch 7, Loss: 0.5100557804107666
Epoch 8, Loss: 0.42246103286743164
Epoch 9, Loss: 0.41318875551223755
Epoch 10, Loss: 0.4926028847694397
Epoch 11, Loss: 0.35688358545303345
Epoch 12, Loss: 0.27402037382125854
Epoch 13, Loss: 0.3495587408542633
Epoch 14, Loss: 0.4036739766597748
Epoch 15, Loss: 0.347205251455307
Epoch 16, Loss: 0.41860759258270264
Epoch 17, Loss: 0.3095571994781494
Epoch 18, Loss: 0.3460979461669922
Epoch 19, Loss: 0.2965511679649353
Epoch 20, Loss: 0.291084885597229
Epoch 21, Loss: 0.29231590032577515
Epoch 22, Loss: 0.28926393389701843
Epoch 23, Loss: 0.17234238982200623
Epoch 24, Loss: 0.19053642451763153
Epoch 25, Loss: 0.3009279668331146
Epoch 26, Loss: 0.2346809208393097
Epoch 27, Loss: 0.22523628175258636
Epoch 28, Loss: 0.220751091837883
Epoch 29, Loss: 0.2671

In [65]:
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)

print("Accuracy:", accuracy)

Accuracy: 0.75


In [66]:
print("Predicted:", predicted[:10])
print("Actual:", y_test[:10])

Predicted: tensor([1, 0, 1, 1])
Actual: tensor([1, 0, 1, 0])


In [67]:
print(len(df))
print(len(X_test))

20
4


In [68]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, predicted))

[[1 1]
 [0 2]]


In [69]:
with zipfile.ZipFile("/content/drive/MyDrive/D/class.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [70]:
import os
print(os.listdir("data"))

['train', 'val', 'data.yaml', 'mimic-iv-clinical-database-demo-2.2', 'test']


In [71]:
import os
print(os.listdir("/content/data/train"))

['pneumonia', 'tuberculosis', 'normal']


In [72]:
import os
import shutil

os.makedirs("train/ABNORMAL", exist_ok=True)

for folder in ["PNEUMONIA", "TUBERCULOSIS"]:
    folder_path = os.path.join("train", folder)

    if os.path.exists(folder_path):
        for file in os.listdir(folder_path):
            shutil.move(os.path.join(folder_path, file), "train/ABNORMAL")

        os.rmdir(folder_path)

In [73]:
for split in ["val", "test"]:
    os.makedirs(f"{split}/ABNORMAL", exist_ok=True)

    for folder in ["PNEUMONIA", "TUBERCULOSIS"]:
        folder_path = os.path.join(split, folder)

        if os.path.exists(folder_path):
            for file in os.listdir(folder_path):
                shutil.move(os.path.join(folder_path, file), f"{split}/ABNORMAL")

            os.rmdir(folder_path)

In [74]:
print(os.listdir("train"))

['ABNORMAL']


In [75]:
import os
print(os.listdir("train"))

['ABNORMAL']


In [76]:
import shutil

shutil.rmtree("train")
shutil.rmtree("val")
shutil.rmtree("test")

In [77]:
with zipfile.ZipFile("/content/drive/MyDrive/D/class.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [78]:
import os
print(os.listdir("/content/data/train"))

['pneumonia', 'tuberculosis', 'normal']


In [79]:
import os
import shutil

base = "data/train"

# Rename normal
if "normal" in os.listdir(base):
    os.rename(f"{base}/normal", f"{base}/NORMAL")

# Create ABNORMAL
os.makedirs(f"{base}/ABNORMAL", exist_ok=True)

# Move disease folders
for folder in ["pneumonia", "tuberculosis"]:
    folder_path = os.path.join(base, folder)

    if os.path.exists(folder_path):
        for file in os.listdir(folder_path):
            shutil.move(os.path.join(folder_path, file), f"{base}/ABNORMAL")

        os.rmdir(folder_path)

In [80]:
base = "data/val"

if "normal" in os.listdir(base):
    os.rename(f"{base}/normal", f"{base}/NORMAL")

os.makedirs(f"{base}/ABNORMAL", exist_ok=True)

for folder in ["pneumonia", "tuberculosis"]:
    folder_path = os.path.join(base, folder)

    if os.path.exists(folder_path):
        for file in os.listdir(folder_path):
            shutil.move(os.path.join(folder_path, file), f"{base}/ABNORMAL")

        os.rmdir(folder_path)

In [81]:
base = "data/test"

if "normal" in os.listdir(base):
    os.rename(f"{base}/normal", f"{base}/NORMAL")

os.makedirs(f"{base}/ABNORMAL", exist_ok=True)

for folder in ["pneumonia", "tuberculosis"]:
    folder_path = os.path.join(base, folder)

    if os.path.exists(folder_path):
        for file in os.listdir(folder_path):
            shutil.move(os.path.join(folder_path, file), f"{base}/ABNORMAL")

        os.rmdir(folder_path)

In [82]:
print(os.listdir("data/train"))
print(os.listdir("data/val"))
print(os.listdir("data/test"))

['ABNORMAL', 'NORMAL']
['ABNORMAL', 'NORMAL']
['ABNORMAL', 'NORMAL']


In [83]:
import shutil

for split in ["train", "val", "test"]:
    path = f"data/{split}/.ipynb_checkpoints"
    if os.path.exists(path):
        shutil.rmtree(path)

In [84]:
print(os.listdir("data/train"))
print(os.listdir("data/val"))
print(os.listdir("data/test"))

['ABNORMAL', 'NORMAL']
['ABNORMAL', 'NORMAL']
['ABNORMAL', 'NORMAL']


In [85]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = ImageFolder("data/train", transform=transform)
val_dataset = ImageFolder("data/val", transform=transform)
test_dataset = ImageFolder("data/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

print(train_dataset.classes)

['ABNORMAL', 'NORMAL']


In [86]:
import torchvision.models as models
import torch.nn as nn

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(512, 2)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 75.7MB/s]


In [87]:
import torch
print(torch.cuda.is_available())

False


In [88]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [89]:
print(device)

cpu


In [90]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

In [ ]:
import torch

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(10):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)   # ✅ move to GPU
        labels = labels.to(device)   # ✅ move to GPU

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print("Test Accuracy:", accuracy)

In [ ]:
class EHRModel(nn.Module):
    def __init__(self):
        super(EHRModel, self).__init__()
        self.feature = nn.Sequential(
            nn.Linear(6, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )

    def forward(self, x):
        return self.feature(x)

In [ ]:
import torchvision.models as models

cnn = models.resnet18(pretrained=True)
cnn.fc = nn.Identity()  # remove final layer

In [ ]:
class MultiModalModel(nn.Module):
    def __init__(self, ehr_model, cnn_model):
        super(MultiModalModel, self).__init__()
        self.ehr = ehr_model
        self.cnn = cnn_model

        self.classifier = nn.Sequential(
            nn.Linear(64 + 512, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, ehr_data, image):
        ehr_features = self.ehr(ehr_data)
        img_features = self.cnn(image)

        combined = torch.cat((ehr_features, img_features), dim=1)
        return self.classifier(combined)

In [ ]:
model = MultiModalModel(EHRModel(), cnn).to(device)

In [ ]:
image_batch = next(iter(train_loader))[0]
ehr_batch = X_train[:len(image_batch)]
labels_batch = y_train[:len(image_batch)]

In [ ]:
for epoch in range(5):
    model.train()

    for i, (images, _) in enumerate(train_loader):
        images = images.to(device)

        # Match EHR batch properly
        start = i * images.size(0)
        end = start + images.size(0)

        ehr_batch = X_train[start:end].to(device)
        labels = y_train[start:end].to(device)

        # Skip if mismatch (last batch safety)
        if ehr_batch.size(0) != images.size(0):
            continue

        outputs = model(ehr_batch, images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)

        batch_size = images.size(0)

        # Force exact matching size
        ehr_batch = X_test[:batch_size].clone().detach().to(device)
        labels = y_test[:batch_size].clone().detach().to(device)

        # Safety check (VERY IMPORTANT)
        if ehr_batch.size(0) != images.size(0):
            min_size = min(ehr_batch.size(0), images.size(0))
            ehr_batch = ehr_batch[:min_size]
            images = images[:min_size]
            labels = labels[:min_size]

        outputs = model(ehr_batch, images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print("Multimodal Accuracy:", accuracy)

Objective 2:

In [ ]:
!pip install transformers

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model = AutoModelForSequenceClassification.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT",
    num_labels=2
)

In [ ]:
texts = [
    "Patient has chest pain and shortness of breath",
    "High fever and persistent cough",
    "Normal condition, no symptoms",
    "Irregular heartbeat and fatigue",
    "Severe headache and dizziness",
    "Low blood pressure and weakness",
    "Stable condition, no issues"
]

labels = [1, 1, 0, 1, 1, 1, 0]

In [ ]:
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

labels = torch.tensor(labels)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}
labels = labels.to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

for epoch in range(5):
    model.train()

    outputs = model(**inputs, labels=labels)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=1)

print("Predictions:", predictions)
print("Actual:", labels)

In [ ]:
new_text = ["Patient experiencing chest pain and high blood pressure"]

new_inputs = tokenizer(
    new_text,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

outputs = model(**new_inputs)
prediction = torch.argmax(outputs.logits, dim=1)

print("Prediction:", prediction.item())

Objective 3:

In [ ]:
import pandas as pd

df = pd.read_csv("/content/data/LUSCexpfile.csv", sep=";")
print(df.shape)

In [ ]:
df = df.set_index(df.columns[0])  # first column = gene names

In [ ]:
df = df.T
print(df.shape)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df['label'] = 1  # all cancer

In [ ]:
df = df.apply(pd.to_numeric, errors='coerce')

In [ ]:
df = df.fillna(0)

In [ ]:
df = df.astype(float)

In [ ]:
import numpy as np

normal_data = df.iloc[:100].copy()

# ✅ Now this will work
normal_data.iloc[:, :] = normal_data.iloc[:, :] * np.random.uniform(0.8, 1.2)

normal_data['label'] = 0
df['label'] = 1

df = pd.concat([df, normal_data], ignore_index=True)

In [ ]:
print(df.dtypes.head())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
import torch

X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test = torch.tensor(X_test.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

In [ ]:
import torch.nn as nn

class GenomicModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

model = GenomicModel(X_train.shape[1])

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(30):
    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)

    accuracy = (predicted == y_test).sum().item() / len(y_test)

print("Genomic Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predicted)
print(cm)